# MobileViT-XS: standalone transfer-learning reference

Uses the same audited 3,600-image bundle and seeds (42, 1337, 2026) as CrackFusion-Tiny.
Run all cells with `RUN_TRAINING=False` to prepare and check the model. No long training starts until
you explicitly enable that flag. Colab mounts `MyDrive/Fridge/major_project_dataset`; local runs
use the repository dataset. The only extra package is `timm` (installed automatically in Colab if missing).

Loads official `mobilevit_xs.cvnets_in1k` ImageNet weights. Input size, interpolation and normalization
come from the checkpoint configuration (256 pixels, rather than CrackFusion's 224). Augmentation
strength, effective batch 64, weighted BCE, validation macro-F1 selection and three seeds are recorded.
The recipe uses five head-only epochs (backbone BatchNorm frozen), followed by up to 50 fine-tuning
epochs; early stopping has patience 15 after 30 total epochs. Checkpoints include phase, optimizer,
AMP, history and random states; all results save under Drive `outputs/mobilevit_baseline`.

This is a practical pretrained reference with a different parameter count and input recipe.
It does not isolate the causal benefit of fusion. Results remain development validation only.
Shared cells are generated from notebook 03 by `scripts/prepare_notebooks.py`.


In [ ]:
import platform, os, csv, hashlib, io, json, math, random, statistics, shutil, sys, time
from pathlib import Path
from tempfile import TemporaryDirectory
import numpy as np
import PIL, sklearn
import matplotlib.pyplot as plt
from PIL import Image
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset
import torchvision
from zipfile import ZipFile
from torchvision import transforms as T
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, confusion_matrix,
    f1_score, precision_score, recall_score, roc_auc_score, average_precision_score)

DRIVE_DATA_DIR = None
try:
    from google.colab import drive
except ImportError:
    pass
else:
    if not Path("/content/drive/MyDrive").is_dir():
        drive.mount("/content/drive")
    DRIVE_DATA_DIR = Path("/content/drive/MyDrive/Fridge/major_project_dataset")
    if not DRIVE_DATA_DIR.is_dir():
        raise FileNotFoundError("Mount the Google account containing MyDrive/Fridge/major_project_dataset.")

PROJECT_ROOT = DRIVE_DATA_DIR or next(
    (p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'notebooks').is_dir() and (p / 'requirements.txt').is_file()),
    Path.cwd())

RUN_SOFTWARE_CHECKS = False  # Enable only to run the optional synthetic checks.
validation_report = {}

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

DEVICE = torch.device('cuda' if torch.cuda.is_available() else
                      'mps' if torch.backends.mps.is_available() else 'cpu')
torch.set_num_threads(min(4, torch.get_num_threads()))
seed_everything(42)
print({'torch': str(torch.__version__), 'torchvision': str(torchvision.__version__),
       'device': str(DEVICE), 'cpu_threads': torch.get_num_threads()})


## Official pretrained model and buffer-safe smoke check


In [ ]:
# Keep the official pretrained download cache in Drive for runtime recovery.
if DRIVE_DATA_DIR:
    os.environ.setdefault('HF_HOME', str(DRIVE_DATA_DIR / 'outputs/pretrained_cache'))
    from importlib.metadata import version, PackageNotFoundError
    try:
        compatible = version('timm') == '1.0.20' and version('huggingface-hub').split('.')[0] == '0'
    except PackageNotFoundError:
        compatible = False
    if not compatible:
        import subprocess
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'timm==1.0.20', 'huggingface-hub>=0.20,<1'])
import timm
from timm.data import resolve_model_data_config, create_transform

MODEL_NAME = 'mobilevit_xs.cvnets_in1k'
ARCHITECTURE = 'MobileViT-XS-timm'
OUTPUT_FAMILY = 'mobilevit_baseline'
PRETRAINED = True

def make_model(pretrained=PRETRAINED):
    return timm.create_model(MODEL_NAME, pretrained=pretrained, num_classes=1)

seed_everything(42)
model = make_model().eval()
DATA_CONFIG = resolve_model_data_config(model)
PARAMETERS = sum(p.numel() for p in model.parameters())
# Fingerprint the loaded pretrained backbone, excluding the newly randomized classifier.
classifier_ids = {id(p) for p in model.get_classifier().parameters()}
classifier_names = {name for name,p in model.named_parameters() if id(p) in classifier_ids}
digest = hashlib.sha256()
for name,value in model.state_dict().items():
    if name not in classifier_names:
        digest.update(name.encode())
        digest.update(value.detach().cpu().contiguous().numpy().tobytes())
PRETRAINED_SHA256 = digest.hexdigest()
buffers = {name:value.clone() for name,value in model.named_buffers()}
with torch.inference_mode():
    logits = model(torch.zeros(2, *DATA_CONFIG['input_size']))
assert logits.shape == (2,1) and torch.isfinite(logits).all()
assert all(torch.equal(buffers[name], value) for name,value in model.named_buffers()), 'Smoke check changed pretrained buffers'
print({'model':MODEL_NAME,'parameters':PARAMETERS,'data_config':DATA_CONFIG,
       'pretrained_backbone_sha256':PRETRAINED_SHA256,'smoke_check':'passed; BN buffers unchanged'})
del model, buffers


## Audit bundle and configuration


In [ ]:
def prepare_colab_data(base):
    audit_dir = base / "outputs" / "data_audit"
    candidates = [base / "crackfusion_colab_bundle.zip",
                  audit_dir / "crackfusion_colab_bundle.zip",
                  Path("/content/crackfusion_colab_bundle.zip")]
    source = next((p for p in candidates if p.is_file()), None)
    cache = Path("/content/crackfusion_data")
    cache.mkdir(parents=True, exist_ok=True)
    manifest = cache / "development_manifest.csv"
    data_root = cache / "dataset" / "SDNET2018"
    if source is not None:
        local_zip = cache / "bundle.zip"
        # Copy the small bundle once; image reads during training stay on local disk.
        if source != local_zip:
            temporary = local_zip.with_suffix(".zip.tmp")
            shutil.copyfile(source, temporary)
            if temporary.stat().st_size != source.stat().st_size:
                raise IOError("Incomplete bundle copy; rerun this cell.")
            temporary.replace(local_zip)
        digest = hashlib.sha256(local_zip.read_bytes()).hexdigest()
        marker = cache / ".extracted_sha256"
        if not marker.is_file() or marker.read_text() != digest:
            with ZipFile(local_zip) as bundle:
                root = cache.resolve()
                for member in bundle.infolist():
                    if not (cache / member.filename).resolve().is_relative_to(root):
                        raise ValueError(f"Unsafe bundle path: {member.filename}")
                bundle.extractall(cache)
            marker.write_text(digest)
    else:
        original_manifest = audit_dir / "development_manifest.csv"
        if not original_manifest.is_file():
            raise FileNotFoundError(
                f"Audit export is missing in {base}. Save crackfusion_colab_bundle.zip "
                "there from notebook 01, or upload that bundle to /content. "
                "Saved notebook output alone does not guarantee that Drive finished syncing.")
        shutil.copyfile(original_manifest, manifest)
        # Audit paths may point at a different computer; find the actual Drive dataset.
        roots = [base / "SDNET2018" / "SDNET2018", base / "SDNET2018", base / "dataset" / "SDNET2018"]
        source_root = next((r for r in roots if (r / "D" / "CD").is_dir()), None)
        if source_root is None:
            raise FileNotFoundError("No extracted SDNET2018 dataset or training bundle in the Drive folder.")
        with manifest.open(newline="") as stream:
            rows = list(csv.DictReader(stream))
        for index, row in enumerate(rows):
            relative = Path(row["path"])
            source_image, target = (source_root / relative).resolve(), (data_root / relative).resolve()
            if not source_image.is_relative_to(source_root.resolve()) or not target.is_relative_to(data_root.resolve()):
                raise ValueError("Unsafe manifest path")
            expected = row.get("sha256", "")
            if not target.is_file() or hashlib.sha256(target.read_bytes()).hexdigest() != expected:
                target.parent.mkdir(parents=True, exist_ok=True)
                temporary = target.with_suffix(target.suffix + ".tmp")
                shutil.copyfile(source_image, temporary)
                temporary.replace(target)
            if (index + 1) % 500 == 0:
                print(f"Cached {index + 1}/{len(rows)} images")
    if not manifest.is_file() or not data_root.is_dir():
        raise FileNotFoundError("Bundle must contain development_manifest.csv and dataset/SDNET2018 images.")
    with manifest.open(newline="") as stream:
        rows = list(csv.DictReader(stream))
    counts = {split: sum(r["split"] == split for r in rows) for split in ("dev_train", "dev_validation")}
    if counts != {"dev_train": 2880, "dev_validation": 720} or len(rows) != 3600:
        raise ValueError(f"Unexpected audit development split: {counts}; expected 2880 train / 720 validation.")
    if any(r.get("split_basis") != "tile_development_only" or not r.get("sha256") for r in rows):
        raise ValueError("Use the hashed development manifest exported by notebook 01.")
    summary_path = audit_dir / "data_audit_summary.json"
    if summary_path.is_file():
        summary = json.loads(summary_path.read_text())
        if summary.get("development_images") != len(rows):
            raise ValueError("Audit summary and bundle disagree on development image count.")
        original = audit_dir / "development_manifest.csv"
        if original.is_file() and original.read_bytes() != manifest.read_bytes():
            raise ValueError("Drive audit manifest and bundle differ. Re-export the bundle from notebook 01.")
    print("Audit split:", counts, "(tile_development_only)")
    print("Local images:", data_root)
    return data_root, manifest



In [ ]:
if DRIVE_DATA_DIR:
    DATA_ROOT, MANIFEST = prepare_colab_data(DRIVE_DATA_DIR)
else:
    roots = [PROJECT_ROOT / 'dataset/SDNET2018/SDNET2018', PROJECT_ROOT / 'dataset/SDNET2018', PROJECT_ROOT / 'dataset']
    DATA_ROOT = next((p for p in roots if (p / 'D/CD').is_dir()), roots[0])
    MANIFEST = PROJECT_ROOT / 'outputs/data_audit/development_manifest.csv'
OUTPUT_ROOT = PROJECT_ROOT / 'outputs' / OUTPUT_FAMILY
PROTOCOL = 'development'
RUN_TRAINING = False  # Enable explicitly after the preflight passes.
PARENT_ID_PROVENANCE = ''
RUN_TEST = False
SEEDS = (42, 1337, 2026)
BATCH_SIZE = 16
EFFECTIVE_BATCH = 64
MAX_EPOCHS = 55
HEAD_EPOCHS = 5
HEAD_LEARNING_RATE = 1e-3
WARMUP_EPOCHS = 5
MIN_EPOCHS = 30
PATIENCE = 15
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-2
THRESHOLD = 0.5


In [ ]:
TRAINING_SOURCE_SHA256 = '02d6b112632429aafa4f9ade9c153a4f68644db258e8c567ae2ae0af6c3851fd'


In [ ]:
def save_json(path, value):
    temporary = path.with_name(path.name + '.tmp')
    with temporary.open('w', encoding='utf-8') as stream:
        json.dump(value, stream, indent=2)
        stream.flush()
        os.fsync(stream.fileno())
    temporary.replace(path)

def save_checkpoint(path, value):
    temporary = path.with_name(path.name + '.tmp')
    with temporary.open('wb') as stream:
        torch.save(value, stream)
        stream.flush()
        os.fsync(stream.fileno())
    temporary.replace(path)



In [ ]:
def load_manifest(path, root, protocol, provenance=''):
    if protocol not in {'source_separated', 'development'}:
        raise ValueError('Unknown protocol')
    with Path(path).open(newline='') as stream:
        reader = csv.DictReader(stream)
        if not {'path', 'label', 'surface', 'parent_id', 'split'} <= set(reader.fieldnames or []):
            raise ValueError('Manifest needs path,label,surface,parent_id,split')
        rows = list(reader)
    if not rows:
        raise ValueError('Empty manifest')
    root = Path(root).resolve()
    seen_paths, hash_splits, parent_splits = set(), {}, {}
    for row in rows:
        split_aliases = {'val': 'validation'}
        if protocol == 'development':
            split_aliases.update(dev_train='train', dev_validation='validation')
        row['split'] = split_aliases.get(row['split'], row['split'])
        if row['split'] not in {'train', 'validation', 'test'} or row['label'] not in {'0', '1'}:
            raise ValueError('Invalid split or binary label')
        if row['surface'] not in {'deck', 'wall', 'pavement'}:
            raise ValueError('Unknown surface')
        image_path = (root / row['path']).resolve()
        if not image_path.is_relative_to(root) or not image_path.is_file():
            raise ValueError(f'Image missing or outside DATA_ROOT: {image_path}')
        if image_path in seen_paths:
            raise ValueError('Repeated image path')
        seen_paths.add(image_path)
        digest = hashlib.sha256(image_path.read_bytes()).hexdigest()
        if row.get('sha256') and digest != row['sha256']:
            raise ValueError(f"Image differs from audit hash: {row['path']}")
        if row.get('excluded', '').lower() in {'true', '1'}:
            raise ValueError('Excluded audit image in training manifest')
        previous = hash_splits.setdefault(digest, row['split'])
        if previous != row['split']:
            raise ValueError('Exact duplicate image crosses splits')
        with Image.open(image_path) as image:
            image.verify()
        parent = row['parent_id'].strip()
        if protocol == 'source_separated':
            if not parent or not provenance.strip():
                raise ValueError('Verified parent IDs and provenance are required')
            if parent_splits.setdefault(parent, row['split']) != row['split']:
                raise ValueError(f'Parent leakage: {parent}')
        row['label'] = int(row['label'])
    groups = {s: [r for r in rows if r['split'] == s] for s in ('train', 'validation', 'test')}
    required = ('train', 'validation', 'test') if protocol == 'source_separated' else ('train', 'validation')
    for split in required:
        if {r['label'] for r in groups[split]} != {0, 1}:
            raise ValueError(f'{split} must contain both classes')
    return groups

class CrackDataset(Dataset):
    def __init__(self, rows, root, transform):
        self.rows, self.root, self.transform = rows, Path(root), transform
    def __len__(self):
        return len(self.rows)
    def __getitem__(self, index):
        row = self.rows[index]
        with Image.open(self.root / row['path']) as image:
            tensor = self.transform(image.convert('RGB'))
        return tensor, torch.tensor([row['label']], dtype=torch.float32)

def training_normalization(rows, root, checkpoint_path=None):
    transform = T.Compose([T.Resize(256), T.CenterCrop(224), T.ToTensor()])
    loader = DataLoader(CrackDataset(rows, root, transform), batch_size=BATCH_SIZE)
    sums, squares, pixels = torch.zeros(3, dtype=torch.float64), torch.zeros(3, dtype=torch.float64), 0
    start_batch = 0
    if checkpoint_path is not None and checkpoint_path.exists():
        saved = json.loads(checkpoint_path.read_text())
        sums = torch.tensor(saved['sums'], dtype=torch.float64)
        squares = torch.tensor(saved['squares'], dtype=torch.float64)
        pixels, start_batch = saved['pixels'], saved['next_batch']
    for batch_index, (images, _) in enumerate(loader):
        if batch_index < start_batch:
            continue
        images = images.double()
        sums += images.sum((0, 2, 3))
        squares += images.square().sum((0, 2, 3))
        pixels += images.shape[0] * images.shape[2] * images.shape[3]
        if checkpoint_path is not None:
            save_json(checkpoint_path, dict(sums=sums.tolist(), squares=squares.tolist(),
                                          pixels=pixels, next_batch=batch_index + 1))
    mean = sums / pixels
    std = (squares / pixels - mean.square()).clamp_min(1e-8).sqrt()
    return mean.tolist(), std.tolist()



In [ ]:
def make_transforms(mean, std):
    size = DATA_CONFIG['input_size'][-1]
    interpolation = getattr(T.InterpolationMode, DATA_CONFIG['interpolation'].upper())
    train = T.Compose([T.RandomResizedCrop(size, scale=(0.85,1.0), interpolation=interpolation),
        T.RandomHorizontalFlip(), T.RandomVerticalFlip(),
        T.RandomRotation(15, interpolation=interpolation, fill=tuple(round(m*255) for m in mean)),
        T.ColorJitter(brightness=0.15,contrast=0.15), T.ToTensor(), T.Normalize(mean,std)])
    return train, create_transform(**DATA_CONFIG, is_training=False)


In [ ]:
def binary_metrics(labels, probabilities, threshold=0.5):
    labels, probabilities = np.asarray(labels).reshape(-1), np.asarray(probabilities).reshape(-1)
    predictions = probabilities >= threshold
    tn, fp, fn, tp = confusion_matrix(labels, predictions, labels=[0, 1]).ravel()
    both = len(np.unique(labels)) == 2
    return dict(accuracy=float(accuracy_score(labels, predictions)),
        balanced_accuracy=float(balanced_accuracy_score(labels, predictions)),
        precision=float(precision_score(labels, predictions, zero_division=0)),
        recall=float(recall_score(labels, predictions, zero_division=0)),
        specificity=float(tn / (tn + fp)) if tn + fp else None,
        f1=float(f1_score(labels, predictions, zero_division=0)),
        macro_f1=float(f1_score(labels, predictions, labels=[0, 1], average='macro', zero_division=0)),
        roc_auc=float(roc_auc_score(labels, probabilities)) if both else None,
        average_precision=float(average_precision_score(labels, probabilities)) if both else None,
        confusion_matrix=[[int(tn), int(fp)], [int(fn), int(tp)]])



In [ ]:
@torch.inference_mode()
def evaluate(net, loader, criterion):
    net.eval()
    labels, probabilities, total = [], [], 0.
    for images, targets in loader:
        images, targets = images.to(DEVICE), targets.to(DEVICE)
        logits = net(images)
        total += criterion(logits, targets).item() * len(images)
        labels.extend(targets.cpu().flatten().tolist())
        probabilities.extend(logits.sigmoid().cpu().flatten().tolist())
    return total / len(loader.dataset), binary_metrics(labels, probabilities, THRESHOLD), probabilities


In [ ]:
def validate_completed_seed(run_dir, seed, groups, result):
    required = [run_dir / f'{prefix}_seed_{seed}.{suffix}' for prefix,suffix in
        [('best','pt'),('last','pt'),('history','json'),('predictions','csv')]]
    if any(not path.is_file() for path in required):
        raise FileNotFoundError(f'Incomplete saved seed {seed}; inspect its artifacts before resuming')
    expected = groups[result['split']]
    with (run_dir / f'predictions_seed_{seed}.csv').open(newline='') as stream:
        rows = list(csv.DictReader(stream))
    if len(rows) != len(expected) or any(any(str(r[k]) != str(e[k]) for k in
            ('path','label','surface','parent_id')) for r,e in zip(rows,expected)):
        raise ValueError('Completed predictions do not match the audited partition')
    probabilities = np.array([float(r['probability']) for r in rows])
    if not np.isfinite(probabilities).all() or not ((0 <= probabilities)&(probabilities <= 1)).all():
        raise ValueError('Invalid saved probabilities')
    if any(int(r['prediction']) != int(p >= THRESHOLD) for r,p in zip(rows,probabilities)):
        raise ValueError('Completed predictions use a different threshold')
    actual = binary_metrics([r['label'] for r in expected], probabilities, THRESHOLD)
    for key,value in actual.items():
        if not ((value is None and result['metrics'][key] is None) or
                (value is not None and np.allclose(value, result['metrics'][key], atol=1e-8))):
            raise ValueError(f'Saved metric differs from predictions: {key}')
    saved = torch.load(required[0], map_location='cpu', weights_only=True)
    if saved.get('training_source_sha256') != TRAINING_SOURCE_SHA256 or saved['seed'] != seed:
        raise ValueError('Completed checkpoint provenance differs')

def step_optimizer(net, optimizer, scaler):
    scaler.unscale_(optimizer)
    gradients = [p.grad for p in net.parameters() if p.grad is not None]
    finite = bool(torch.stack([torch.isfinite(g).all() for g in gradients]).all().item())
    if finite:
        nn.utils.clip_grad_norm_(net.parameters(), 1.0, error_if_nonfinite=True)
    elif not scaler.is_enabled():
        raise FloatingPointError('Nonfinite gradients with AMP disabled')
    # unscale_ recorded any overflow: GradScaler skips the update and lowers its scale.
    scaler.step(optimizer)
    scaler.update()
    optimizer.zero_grad(set_to_none=True)
    return finite

def train_one_seed(groups, mean, std, seed, run_dir):
    seed_everything(seed)
    train_tf, eval_tf = make_transforms(mean, std)
    loaders = {split: DataLoader(CrackDataset(rows, DATA_ROOT,
                    train_tf if split == 'train' else eval_tf), batch_size=BATCH_SIZE,
                    shuffle=split == 'train', num_workers=0,
                    generator=torch.Generator().manual_seed(seed))
               for split, rows in groups.items() if rows}
    resume_path = run_dir / f'last_seed_{seed}.pt'
    net = make_model(pretrained=PRETRAINED and not resume_path.exists()).to(DEVICE)
    positive = sum(r['label'] for r in groups['train'])
    weight = (len(groups['train']) - positive) / positive
    criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([weight], device=DEVICE))
    phase = 'head' if HEAD_EPOCHS else 'full'
    def phase_optimizer(phase):
        for parameter in net.parameters():
            parameter.requires_grad_(phase == 'full')
        if phase == 'head':
            for parameter in net.get_classifier().parameters():
                parameter.requires_grad_(True)
        return torch.optim.AdamW([p for p in net.parameters() if p.requires_grad],
            lr=HEAD_LEARNING_RATE if phase == 'head' else LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    optimizer = phase_optimizer(phase)
    scaler = torch.amp.GradScaler('cuda', enabled=DEVICE.type == 'cuda')
    accumulation = EFFECTIVE_BATCH // BATCH_SIZE
    best, stale, history = -1., 0, []
    checkpoint_path = run_dir / f'best_seed_{seed}.pt'
    resume_path = run_dir / f'last_seed_{seed}.pt'
    start_epoch = 0
    if resume_path.exists():
        saved = torch.load(resume_path, map_location='cpu', weights_only=True)
        if saved.get('training_source_sha256') != TRAINING_SOURCE_SHA256:
            raise ValueError('Training code differs from the saved checkpoint')
        phase = saved['phase']
        optimizer = phase_optimizer(phase)
        net.load_state_dict(saved['model'])
        optimizer.load_state_dict(saved['optimizer'])
        scaler.load_state_dict(saved['scaler'])
        start_epoch, best, stale, history = saved['next_epoch'], saved['best'], saved['stale'], saved['history']
        random.setstate(saved['python_rng'])
        numpy_rng = saved['numpy_rng']
        np.random.set_state((numpy_rng[0], np.asarray(numpy_rng[1], dtype=np.uint32), *numpy_rng[2:]))
        torch.set_rng_state(saved['torch_rng'])
        if DEVICE.type == 'cuda' and saved['cuda_rng']:
            torch.cuda.set_rng_state_all(saved['cuda_rng'])
        for split, state in saved['loader_rng'].items():
            loaders[split].generator.set_state(state)
        print(f'Resuming seed={seed} after epoch={start_epoch}')
    for epoch in range(start_epoch, MAX_EPOCHS):
        if epoch >= max(MIN_EPOCHS, HEAD_EPOCHS + 1) and stale >= PATIENCE:
            break
        next_phase = 'head' if epoch < HEAD_EPOCHS else 'full'
        if next_phase != phase:
            phase = next_phase
            optimizer = phase_optimizer(phase)
            stale = 0
        phase_epoch = epoch if phase == 'head' else epoch - HEAD_EPOCHS
        phase_length = HEAD_EPOCHS if phase == 'head' else MAX_EPOCHS - HEAD_EPOCHS
        factor = (1.0 if phase == 'head' else
            (phase_epoch + 1) / WARMUP_EPOCHS if phase_epoch < WARMUP_EPOCHS else
            0.5 * (1 + math.cos(math.pi * (phase_epoch - WARMUP_EPOCHS) /
                max(1, phase_length - WARMUP_EPOCHS - 1))))
        for group in optimizer.param_groups:
            group['lr'] = (HEAD_LEARNING_RATE if phase == 'head' else LEARNING_RATE) * factor
        net.train()
        if phase == 'head':
            net.eval()  # Keep the frozen backbone's BatchNorm and dropout unchanged.
            net.get_classifier().train()
        optimizer.zero_grad(set_to_none=True)
        total = 0.
        amp_skipped = 0
        for index, (images, targets) in enumerate(loaders['train']):
            images, targets = images.to(DEVICE), targets.to(DEVICE)
            start = (index // accumulation) * accumulation * BATCH_SIZE
            group_size = min(EFFECTIVE_BATCH, len(groups['train']) - start)
            with torch.autocast(device_type=DEVICE.type, enabled=DEVICE.type == 'cuda'):
                loss = criterion(net(images), targets)
            if not torch.isfinite(loss):
                raise FloatingPointError('Nonfinite training loss')
            scaler.scale(loss * len(images) / group_size).backward()
            if (index + 1) % accumulation == 0 or index + 1 == len(loaders['train']):
                if not step_optimizer(net, optimizer, scaler):
                    amp_skipped += 1
                    print(f'AMP overflow: skipped update; scale reduced to {scaler.get_scale():g}')
            total += loss.item() * len(images)
        val_loss, metrics, _ = evaluate(net, loaders['validation'], criterion)
        record = dict(epoch=epoch+1, phase=phase, amp_skipped_updates=amp_skipped, train_loss=total/len(groups['train']),
                      validation_loss=val_loss, lr=optimizer.param_groups[0]['lr'], **metrics)
        history.append(record)
        print(f"seed={seed} epoch={epoch+1} loss={record['train_loss']:.4f} val_macro_f1={metrics['macro_f1']:.4f}")
        if metrics['macro_f1'] > best:
            best, stale = metrics['macro_f1'], 0
            save_checkpoint(checkpoint_path, dict(model=net.state_dict(), seed=seed, epoch=epoch+1,
                architecture=ARCHITECTURE, model_name=MODEL_NAME, parameters=PARAMETERS,
                training_source_sha256=TRAINING_SOURCE_SHA256, data_config=DATA_CONFIG,
                mean=mean, std=std, threshold=THRESHOLD, protocol=PROTOCOL,
                validation_metrics=metrics))
        else:
            stale += 1
        numpy_rng = np.random.get_state()
        save_checkpoint(resume_path, dict(model=net.state_dict(), optimizer=optimizer.state_dict(),
            scaler=scaler.state_dict(), phase=phase, training_source_sha256=TRAINING_SOURCE_SHA256, next_epoch=epoch + 1, best=best, stale=stale, history=history,
            python_rng=random.getstate(),
            numpy_rng=(numpy_rng[0], numpy_rng[1].tolist(), *numpy_rng[2:]),
            torch_rng=torch.get_rng_state(),
            cuda_rng=torch.cuda.get_rng_state_all() if DEVICE.type == 'cuda' else [],
            loader_rng={split: loader.generator.get_state() for split, loader in loaders.items()}))
        save_json(run_dir / f'history_seed_{seed}.json', history)
        if epoch + 1 >= max(MIN_EPOCHS, HEAD_EPOCHS + 1) and stale >= PATIENCE:
            break
    checkpoint = torch.load(checkpoint_path, map_location=DEVICE, weights_only=True)
    net.load_state_dict(checkpoint['model'])
    split = 'test' if RUN_TEST else 'validation'
    _, metrics, probabilities = evaluate(net, loaders[split], criterion)
    per_surface = {}
    for surface in ('deck', 'wall', 'pavement'):
        selected = [(r['label'], p) for r, p in zip(groups[split], probabilities) if r['surface'] == surface]
        if selected:
            ys, ps = zip(*selected)
            per_surface[surface] = binary_metrics(ys, ps, THRESHOLD)
    result = dict(seed=seed, split=split, protocol=PROTOCOL, metrics=metrics, per_surface=per_surface)

    prediction_path = run_dir / f'predictions_seed_{seed}.csv'
    temporary_predictions = prediction_path.with_suffix('.csv.tmp')
    with temporary_predictions.open('w', newline='') as stream:
        writer = csv.DictWriter(stream, fieldnames=['path', 'label', 'surface', 'parent_id', 'probability', 'prediction'])
        writer.writeheader()
        for row, probability in zip(groups[split], probabilities):
            writer.writerow({k: row[k] for k in ('path', 'label', 'surface', 'parent_id')} |
                            dict(probability=probability, prediction=int(probability >= THRESHOLD)))
    temporary_predictions.replace(prediction_path)
    # Mark a seed complete only after its predictions have been written.
    save_json(run_dir / f'metrics_seed_{seed}.json', result)
    return result

def run_experiment():
    if BATCH_SIZE <= 0 or EFFECTIVE_BATCH % BATCH_SIZE or MAX_EPOCHS < 1 or not 0 <= HEAD_EPOCHS < MAX_EPOCHS or WARMUP_EPOCHS < 1:
        raise ValueError('Batch size must divide effective batch; epochs must be positive')
    if RUN_TEST and PROTOCOL != 'source_separated':
        raise ValueError('Development splits cannot produce research test results')
    manifest_bytes = MANIFEST.read_bytes()
    configuration = {key: globals()[key] for key in ('PROTOCOL', 'PARENT_ID_PROVENANCE', 'SEEDS',
        'BATCH_SIZE', 'EFFECTIVE_BATCH', 'MAX_EPOCHS', 'HEAD_EPOCHS', 'HEAD_LEARNING_RATE', 'WARMUP_EPOCHS', 'MIN_EPOCHS',
        'PATIENCE', 'LEARNING_RATE', 'WEIGHT_DECAY', 'THRESHOLD', 'RUN_TEST')}
    configuration.update(parameters=PARAMETERS, architecture=ARCHITECTURE, model_name=MODEL_NAME,
        training_source_sha256=TRAINING_SOURCE_SHA256, pretrained=PRETRAINED,
        pretrained_backbone_sha256=PRETRAINED_SHA256, data_config=DATA_CONFIG,
        timm=str(timm.__version__) if PRETRAINED else None,
        torch=str(torch.__version__), torchvision=str(torchvision.__version__), device=str(DEVICE),
        python=platform.python_version(), numpy=str(np.__version__), pillow=str(PIL.__version__), sklearn=str(sklearn.__version__),
        manifest_sha256=hashlib.sha256(manifest_bytes).hexdigest(), data_root=str(DATA_ROOT.resolve()))
    identity = hashlib.sha256(json.dumps(configuration, sort_keys=True).encode()).hexdigest()[:16]
    run_dir = OUTPUT_ROOT / ('run_' + identity)
    run_dir.mkdir(parents=True, exist_ok=True)
    if (run_dir / 'configuration.json').exists():
        previous = json.loads((run_dir / 'configuration.json').read_text())
        if any(previous.get(key) != json.loads(json.dumps(value)) for key,value in configuration.items()):
            raise ValueError('Existing run configuration differs; refusing to overwrite it')
    save_json(run_dir / 'configuration.json', configuration)
    manifest_temp = run_dir / 'manifest.csv.tmp'
    manifest_temp.write_bytes(manifest_bytes)
    manifest_temp.replace(run_dir / 'manifest.csv')
    print('Checkpoints:', run_dir.resolve())
    groups = load_manifest(MANIFEST, DATA_ROOT, PROTOCOL, PARENT_ID_PROVENANCE)
    save_json(run_dir / 'manifest_verified.json', {'manifest_sha256': configuration['manifest_sha256'],
        'counts': {split: len(rows) for split, rows in groups.items()}})
    if PRETRAINED:
        mean, std = list(DATA_CONFIG['mean']), list(DATA_CONFIG['std'])
    else:
        mean, std = training_normalization(groups['train'], DATA_ROOT, run_dir / 'normalization_progress.json')
    configuration.update(mean=mean, std=std)
    save_json(run_dir / 'configuration.json', configuration)
    results = []
    for seed in SEEDS:
        result_path = run_dir / f'metrics_seed_{seed}.json'
        if result_path.exists():
            result = json.loads(result_path.read_text())
            validate_completed_seed(run_dir, seed, groups, result)
            print(f'Reusing verified completed seed={seed}')
        else:
            try:
                result = train_one_seed(groups, mean, std, seed, run_dir)
            except BaseException as error:
                save_json(run_dir / 'last_error.json', {'seed': seed, 'error': repr(error),
                    'resume': 'Rerun this cell to continue from the last completed epoch.'})
                raise
        results.append(result)
        save_json(run_dir / 'completed_seeds.json', [result['seed'] for result in results])
    summary = {}
    for key in ('macro_f1', 'f1', 'recall', 'balanced_accuracy', 'average_precision'):
        values = [r['metrics'][key] for r in results]
        summary[key] = dict(mean=statistics.mean(values),
                            seed_std=statistics.stdev(values) if len(values) > 1 else None)
    save_json(run_dir / 'summary.json', summary)
    print('Saved:', run_dir.resolve())
    return results, summary



## Preparation check (no experiment training)


In [ ]:
# This verifies all selected image hashes and one disposable backward/checkpoint step.
# It writes only preflight_report.json; no experiment or training run is started.
groups = load_manifest(MANIFEST, DATA_ROOT, PROTOCOL, PARENT_ID_PROVENANCE)
train_tf, eval_tf = make_transforms(
    list(DATA_CONFIG['mean']) if PRETRAINED else [0.5]*3,
    list(DATA_CONFIG['std']) if PRETRAINED else [0.25]*3)
sample_rows = [next(r for r in groups['train'] if r['label'] == label) for label in (0,1)]
images, targets = next(iter(DataLoader(CrackDataset(sample_rows, DATA_ROOT, train_tf), batch_size=2)))
probe = make_model(pretrained=False).to(DEVICE)
probe.train()
optimizer = torch.optim.AdamW(probe.parameters(), lr=LEARNING_RATE)
scaler = torch.amp.GradScaler('cuda', enabled=DEVICE.type == 'cuda')
with torch.autocast(device_type=DEVICE.type, enabled=DEVICE.type == 'cuda'):
    loss = nn.BCEWithLogitsLoss()(probe(images.to(DEVICE)), targets.to(DEVICE))
assert torch.isfinite(loss)
scaler.scale(loss).backward()
assert step_optimizer(probe, optimizer, scaler)
probe.eval()
with torch.inference_mode():
    expected = probe(images.to(DEVICE)).cpu()
with TemporaryDirectory() as scratch:
    path = Path(scratch) / 'probe.pt'
    save_checkpoint(path, {'model':probe.state_dict(), 'optimizer':optimizer.state_dict(), 'scaler':scaler.state_dict()})
    restored = make_model(pretrained=False).to(DEVICE).eval()
    restored.load_state_dict(torch.load(path,map_location=DEVICE,weights_only=True)['model'])
    with torch.inference_mode():
        torch.testing.assert_close(restored(images.to(DEVICE)).cpu(),expected)
OUTPUT_ROOT.mkdir(parents=True,exist_ok=True)
save_json(OUTPUT_ROOT / 'preflight_report.json', dict(
    model_name=MODEL_NAME, parameters=PARAMETERS, device=str(DEVICE),
    manifest_sha256=hashlib.sha256(MANIFEST.read_bytes()).hexdigest(),
    counts={split:len(rows) for split,rows in groups.items()},
    training_source_sha256=TRAINING_SOURCE_SHA256, pretrained_backbone_sha256=PRETRAINED_SHA256,
    image_hashes='passed', finite_backward='passed', checkpoint_round_trip='passed',
    experiment_training_started=False))
print('Preflight passed. Training is controlled by RUN_TRAINING in the configuration cell.')
del probe, restored, optimizer, scaler, images, targets
seed_everything(42)


## Start or resume training


In [ ]:
if RUN_TRAINING:
    experiment_results, experiment_summary = run_experiment()
    print(json.dumps(experiment_summary,indent=2))
else:
    print('Experiment training disabled. Set RUN_TRAINING=True after reviewing the preflight.')


## Recovery
Rerun with the same recipe to resume the last completed epoch. The head/full transition resets the optimizer once. Notebook 04 reads the same metrics and prediction schema for both models.
